# Degradation pilot (no training)

**Goal:** find the degradation range where the clean-trained baseline (M0) actually loses accuracy, using the models that are **already trained**. Today's worst condition costs M0 only ~0.006 Dice, so no model can show a robustness gain. This picks the new range from evidence before anything is retrained.

**What it runs:** 5 trained models (M0, M1, M4, M5, M6) on the 37 **validation** patients (the test set stays untouched) under 36 conditions: SNR 8/5/3/2, in-plane resolution r 1.0/0.5, slice thickness 1/5 mm, and two noise models (the current pipeline, and the physically correct "noise only in acquired k-space").

**Before you press Run all:**
1. Upload `archive.zip` (the BraTS download, ~4.5 GB) to Google Drive at `MyDrive/Argus/archive.zip`.
2. *Runtime → Change runtime type →* a **GPU** (T4 is enough; A100/L4 is faster on Colab Pro).
3. *Runtime → Run all.* Results are saved to `MyDrive/Argus/pilot/` after every patient, so a disconnect loses nothing: just run all again and it resumes.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---- Configuration ----
ZIP = '/content/drive/MyDrive/Argus/archive.zip'   # BraTS download on Drive
OUT = '/content/drive/MyDrive/Argus/pilot'         # results land here (resumable)
REPO = 'https://github.com/joshykoshy/Argus.git'
BRANCH = 'matlab-analysis'
import os; os.makedirs(OUT, exist_ok=True)

In [ ]:
# Code + the 8 trained checkpoints (stored in git, ~400 MB)
!rm -rf /content/Argus && git clone -q --depth 1 -b {BRANCH} {REPO} /content/Argus
%cd /content/Argus
!pip -q install nibabel
!ls -la results/M0/seed_0 results/M4/seed_0

In [ ]:
# Extract ONLY the 37 validation patients from the zip (to the fast local disk)
import json, zipfile, glob
val = json.load(open('data/splits/split_v1.json'))['val']
with zipfile.ZipFile(ZIP) as z:
    names = [n for n in z.namelist()
             if any(f'/{p}/' in n for p in val) or n.endswith('/name_mapping.csv')]
    z.extractall('/content/brats', members=names)
RAW = glob.glob('/content/brats/**/MICCAI_BraTS2020_TrainingData', recursive=True)[0]
print(RAW, '|', len([d for d in os.listdir(RAW) if d.startswith('BraTS20')]), 'patients extracted')

In [ ]:
# Smoke test: self-checks + ONE patient, to see the time per patient
!python -m pilot.pilot --raw_dir {RAW} --out_csv /content/smoke.csv --max_patients 1

In [ ]:
# Full pilot: all 37 validation patients (resumes if interrupted)
!python -m pilot.pilot --raw_dir {RAW} --out_csv {OUT}/pilot_results.csv

In [ ]:
# Summary table + figure
!python pilot/analyze.py --csv {OUT}/pilot_results.csv --out {OUT}
from IPython.display import Image
Image(f'{OUT}/pilot_dice_vs_snr.png')

**What to bring back:** `pilot_summary.csv` and `pilot_dice_vs_snr.png` in `MyDrive/Argus/pilot/` (plus `pilot_results.csv`, the raw per-patient numbers). Download them into the repo's `results/pilot/` folder, or paste the printed table into the chat.

**How to read it:** look for conditions where **M0 loses at least 0.10 Dice**. That is the range where a robustness method has room to show a gain. The `M4-M1` column is the fair architecture comparison (same training data); the trained models here have one seed, so gaps under ~0.02 are within training noise.

## Diagnostic: why is clean 3D Dice ~0.2?

The pilot's clean row gave 3D Dice ~0.2, but training logs said ~0.75. Training validation scored only the 8 largest-tumor slices per patient, and the models never trained on tumor-free slices. This cell scores the same predictions both ways and counts false tumor on slices that have none.

If your runtime was reset, run the setup cells above first (GPU check, Drive, configuration, clone, extract), then this one. Takes a few minutes.

In [ ]:
!git -C /content/Argus pull -q
!python -m pilot.diagnose --raw_dir {RAW} --out {OUT} --max_patients 10
from IPython.display import Image
Image(f'{OUT}/diagnose_false_positives.png')